To plot the "cliff" distance vs height for different combinations of USI and MCS

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

def find_cliff_hdist(df, threshold):
    """
    Identify the 'cliff' point in the data where the value drops significantly.
    The cliff is defined as the horizontal distance where the reliability drops below the threshold.

    Parameters:
    df (pd.DataFrame): DataFrame for a combination of height, USI, and Bitrate. The columns are 'Horizontal Distance' and 'Reliability'.

    Returns:
    hdist: The horizontal distance of the cliff point.
    """
    before_cliff = df[df['Reliability'] >= threshold]
    if before_cliff.empty:
        return 0  # No cliff found, all values are below threshold
    return before_cliff['Horizontal_Distance'].max() # Here we are assuming that the reliability vs horizontal distance is essentially monotonic decreasing.

In [4]:
link = "Video" # "Downlink", "Uplink", "Video"
USI = [10, 20, 66.7, 100]
BITRATE = [6.5, 13, 19.5, 26, 39, 52, 58.5, 65]
HEIGHT = [60, 90, 120, 150, 180, 210, 240, 270, 300]
# Training Dataset
dataset_file_path = f"/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/{link}_Reliability.csv"
save_path = f"/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/cliff_vs_height_4_training_data/{link.lower()}/"

os.makedirs(save_path, exist_ok=True)
data_df = pd.read_csv(dataset_file_path)
non_viable_combinations = [] # To store combinations with no region that are reliable enough

if "Num_Sent" in data_df.columns and "Num_Reliable" in data_df.columns:
    data_df["Reliability"] = data_df["Num_Reliable"] / data_df["Num_Sent"]
elif "Num_Fail_Other" in data_df.columns and "Num_Delay_Excd" in data_df.columns and "Num_Reliable" in data_df.columns:
    data_df["Reliability"] = data_df["Num_Reliable"] / (data_df["Num_Reliable"] + data_df["Num_Delay_Excd"] + data_df["Num_Fail_Other"])
    
for usi in USI:
    for bitrate in BITRATE:
        cliff_hdist_list = []
        for height in HEIGHT:
            subset = data_df[(data_df['UAV_Sending_Interval'] == usi) & (data_df['Bitrate'] == bitrate) & (data_df['Height'] == height)]
            cliff_hdist_list.append(find_cliff_hdist(subset, threshold=0.99))
        # Check that any cliff points are not 0
        if any(hdist > 0 for hdist in cliff_hdist_list):
            plt.figure(figsize=(10, 4))
            plt.rcParams.update({'font.size': 14})
            plt.plot(HEIGHT, cliff_hdist_list, label=f'USI={usi}, Bitrate={bitrate}Mbps')
            plt.ylim(0, max(cliff_hdist_list) * 1.1)
            plt.ylabel("Cliff Horizontal Distance (m)")
            plt.xlabel("Height (m)")
            plt.title(f"Link: {link}, USI: {usi} ms, Bitrate: {bitrate} Mbps")
            # Save the plot with a descriptive filename
            filename = f"Cliff_vs_Height_USI-{usi}_Bitrate-{bitrate}.png"
            plt.grid(True)
            plt.savefig(os.path.join(save_path, filename))
            plt.close()
        else:
            non_viable_combinations.append((usi, bitrate))

In [7]:
# Print non-viable combinations
print("Non-viable combinations (USI, Bitrate):")
for combo in non_viable_combinations:
    print(combo)

Non-viable combinations (USI, Bitrate):
(10, 6.5)
(10, 13)
(20, 6.5)
